# ⚡ Dashboard CSR — Jabatan Kejuruteraan Elektrik
## Politeknik Kota Bharu

**Arahan penggunaan:**
1. Jalankan **Sel 1** untuk pasang pakej
2. Jalankan **Sel 2** untuk muat naik fail CSV anda (Program, Ahli, Kehadiran, Kewangan)
3. Jalankan **Sel 3 hingga akhir** untuk papar dashboard
4. Jalankan **Sel Terakhir** untuk muat turun `dashboard.html`

> 💡 Cara paling mudah: Klik **Runtime → Run all** untuk jalankan semua sel sekaligus

In [ ]:
# ── Sel 1: Pasang pakej yang diperlukan ──────────────────────────────────────
# Colab sudah ada pandas dan plotly — sel ini pastikan versi terkini
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'plotly>=5.22', 'pandas>=2.0'])
print('✅ Pakej berjaya dipasang/dikemas kini')

In [ ]:
# ── Sel 2: Muat naik fail CSV ─────────────────────────────────────────────────
# Jalankan sel ini → klik butang 'Choose Files' → pilih KEEMPAT-EMPAT fail CSV
# (Program.csv, Ahli.csv, Kehadiran.csv, Kewangan.csv)

from google.colab import files as colab_files
import io, os

print('📂 Sila pilih fail CSV anda (boleh pilih banyak sekaligus)...')
uploaded = colab_files.upload()

fail_dimuat = list(uploaded.keys())
print(f'\n✅ Fail yang berjaya dimuat naik: {", ".join(fail_dimuat)}')

# Semak fail yang diperlukan
FAIL_WAJIB = ['Program.csv', 'Ahli.csv', 'Kehadiran.csv', 'Kewangan.csv']
fail_hilang = [f for f in FAIL_WAJIB if f not in fail_dimuat]
if fail_hilang:
    print(f'⚠️  Fail berikut belum dimuat naik: {", ".join(fail_hilang)}')
    print('   Jalankan semula sel ini untuk muat naik fail yang tertinggal.')
else:
    print('✅ Semua fail CSV telah dimuat naik!')

In [ ]:
# ── Sel 3: Tetapan dan pemalar ────────────────────────────────────────────────
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

# Nilai sah
STATUS_SAH          = ['Dirancang', 'Sedang Berjalan', 'Selesai', 'Dibatalkan']
KATEGORI_SAH        = ['Pelajar', 'Pensyarah']
JENIS_KEWANGAN_SAH  = ['Peruntukan', 'Sumbangan Luar', 'Perbelanjaan']
SASARAN_PROGRAM     = 4   # ← Tukar nilai ini mengikut sasaran jabatan
SEMBUNYIKAN_NAMA    = True  # ← True = sembunyikan nama, False = papar nama sebenar

WARNA_STATUS = {
    'Dirancang':       '#3b82f6',
    'Sedang Berjalan': '#f59e0b',
    'Selesai':         '#22c55e',
    'Dibatalkan':      '#ef4444',
}

print('✅ Tetapan dimuat. Sasaran program:', SASARAN_PROGRAM)
print('🔒 Sembunyikan nama:', SEMBUNYIKAN_NAMA)

In [ ]:
# ── Sel 4: Muat dan sahkan CSV ────────────────────────────────────────────────

KOLUM_WAJIB = {
    'Program.csv':   ['ID Program','Nama Program','Tarikh','Lokasi',
                      'Jenis Aktiviti','Status','Rakan Strategik',
                      'Penerima Manfaat','Laporan Dihantar','Pautan Laporan'],
    'Ahli.csv':      ['Bil','Nama','No. Matrik / ID Staf',
                      'Kategori','Program Pengajian','Semester','Peranan'],
    'Kehadiran.csv': ['ID Program','Nama Ahli','Hadir','Jam Sumbangan'],
    'Kewangan.csv':  ['ID Program','Perkara','Jenis','Jumlah (RM)'],
}

ralat = []
amaran = []
df = {}

for nama_fail, kolum_wajib in KOLUM_WAJIB.items():
    if nama_fail not in uploaded:
        ralat.append(f'❌ Fail {nama_fail} tidak dijumpai.')
        df[nama_fail] = pd.DataFrame()
        continue
    try:
        data_fail = pd.read_csv(io.BytesIO(uploaded[nama_fail]),
                                encoding='utf-8-sig', dtype=str)
        data_fail.columns = [c.strip() for c in data_fail.columns]
        for lajur in data_fail.select_dtypes('object').columns:
            data_fail[lajur] = data_fail[lajur].str.strip()
        hilang = [k for k in kolum_wajib if k not in data_fail.columns]
        if hilang:
            ralat.append(f'❌ {nama_fail}: Lajur wajib hilang — {", ".join(hilang)}')
            df[nama_fail] = pd.DataFrame()
        else:
            df[nama_fail] = data_fail
            print(f'✅ {nama_fail} — {len(data_fail)} rekod dimuat')
    except Exception as e:
        ralat.append(f'❌ {nama_fail}: Ralat semasa membaca — {e}')
        df[nama_fail] = pd.DataFrame()

# Normalisasi Program.csv
if not df['Program.csv'].empty:
    p = df['Program.csv']
    p['Tarikh'] = pd.to_datetime(p['Tarikh'], errors='coerce')
    p['Laporan Dihantar'] = p['Laporan Dihantar'].str.capitalize().map(
        {'Ya': True, 'Tidak': False})
    p['Penerima Manfaat'] = pd.to_numeric(
        p['Penerima Manfaat'], errors='coerce').fillna(0).astype(int)
    status_luar = p[~p['Status'].isin(STATUS_SAH)]['Status'].dropna().unique()
    if len(status_luar):
        amaran.append(f'⚠️  Program.csv: Nilai Status tidak dijangka — {", ".join(status_luar)}')
    df['Program.csv'] = p

# Normalisasi Ahli.csv
if not df['Ahli.csv'].empty:
    kat_luar = df['Ahli.csv'][~df['Ahli.csv']['Kategori'].isin(KATEGORI_SAH)]['Kategori'].dropna().unique()
    if len(kat_luar):
        amaran.append(f'⚠️  Ahli.csv: Nilai Kategori tidak dijangka — {", ".join(kat_luar)}')

# Normalisasi Kehadiran.csv
if not df['Kehadiran.csv'].empty:
    k = df['Kehadiran.csv']
    k['Hadir'] = k['Hadir'].str.capitalize().map({'Ya': True, 'Tidak': False})
    k['Jam Sumbangan'] = pd.to_numeric(
        k['Jam Sumbangan'], errors='coerce').fillna(0).clip(lower=0)
    df['Kehadiran.csv'] = k

# Normalisasi Kewangan.csv
if not df['Kewangan.csv'].empty:
    kw = df['Kewangan.csv']
    kw['Jumlah (RM)'] = pd.to_numeric(
        kw['Jumlah (RM)'], errors='coerce').fillna(0)
    jenis_luar = kw[~kw['Jenis'].isin(JENIS_KEWANGAN_SAH)]['Jenis'].dropna().unique()
    if len(jenis_luar):
        amaran.append(f'⚠️  Kewangan.csv: Nilai Jenis tidak dijangka — {", ".join(jenis_luar)}')
    df['Kewangan.csv'] = kw

# Semak ID tidak sepadan
if not df['Program.csv'].empty:
    id_prog = set(df['Program.csv']['ID Program'].dropna())
    for nama_fail in ['Kehadiran.csv', 'Kewangan.csv']:
        if not df[nama_fail].empty:
            tidak_sepadan = set(df[nama_fail]['ID Program'].dropna()) - id_prog
            if tidak_sepadan:
                amaran.append(
                    f'⚠️  {nama_fail}: ID Program tidak sepadan — {", ".join(sorted(tidak_sepadan))}. '
                    f'Rekod ini TIDAK dikira dalam KPI.')

# Papar ringkasan
print()
for r in ralat:  print(r)
for a in amaran: print(a)
if not ralat and not amaran:
    print('✅ Semua data sah dan bersih!')

In [ ]:
# ── Sel 5: Topeng nama (privasi) ──────────────────────────────────────────────

def bina_peta_topeng(nilai, awalan):
    unik = sorted(set(v for v in nilai if pd.notna(v) and str(v).strip()))
    return {nama: f'{awalan}-{str(i+1).zfill(3)}' for i, nama in enumerate(unik)}

ahli_df     = df['Ahli.csv'].copy()
kehadiran_df = df['Kehadiran.csv'].copy()

if SEMBUNYIKAN_NAMA:
    semua_nama = []
    if not ahli_df.empty and 'Nama' in ahli_df.columns:
        semua_nama += ahli_df['Nama'].tolist()
    if not kehadiran_df.empty and 'Nama Ahli' in kehadiran_df.columns:
        semua_nama += kehadiran_df['Nama Ahli'].tolist()

    peta_nama = bina_peta_topeng(semua_nama, 'Ahli')

    if not ahli_df.empty and 'Nama' in ahli_df.columns:
        peta_id = bina_peta_topeng(ahli_df['No. Matrik / ID Staf'].tolist(), 'M')
        ahli_df['Nama'] = ahli_df['Nama'].map(peta_nama).fillna(ahli_df['Nama'])
        ahli_df['No. Matrik / ID Staf'] = ahli_df['No. Matrik / ID Staf'].map(
            peta_id).fillna(ahli_df['No. Matrik / ID Staf'])

    if not kehadiran_df.empty and 'Nama Ahli' in kehadiran_df.columns:
        kehadiran_df['Nama Ahli'] = kehadiran_df['Nama Ahli'].map(
            peta_nama).fillna(kehadiran_df['Nama Ahli'])

    print('🔒 Nama dan No. Matrik telah disembunyikan (Ahli-001, M-001, ...)')
else:
    print('👁️  Nama sebenar dipapar (SEMBUNYIKAN_NAMA = False)')

In [ ]:
# ── Sel 6: Kira semua KPI ─────────────────────────────────────────────────────

prog_df = df['Program.csv']
kew_df  = df['Kewangan.csv']

# Saring kehadiran & kewangan kepada ID program yang sah sahaja
id_sah = set(prog_df['ID Program'].dropna()) if not prog_df.empty else set()

had_df = kehadiran_df[
    kehadiran_df['ID Program'].isin(id_sah)] if not kehadiran_df.empty else kehadiran_df
kew_df_sah = kew_df[
    kew_df['ID Program'].isin(id_sah)] if not kew_df.empty else kew_df

# Pengiraan KPI
kpi_program_selesai = int((prog_df['Status'] == 'Selesai').sum()) \
    if not prog_df.empty else 0

if not ahli_df.empty and 'No. Matrik / ID Staf' in ahli_df.columns:
    unik_ahli = ahli_df.drop_duplicates(subset=['No. Matrik / ID Staf'])
    kpi_pelajar   = int((unik_ahli['Kategori'] == 'Pelajar').sum())
    kpi_pensyarah = int((unik_ahli['Kategori'] == 'Pensyarah').sum())
else:
    kpi_pelajar = kpi_pensyarah = 0

kpi_penerima = int(prog_df['Penerima Manfaat'].sum()) \
    if not prog_df.empty else 0

if not had_df.empty and 'Hadir' in had_df.columns:
    kpi_jam = float(had_df[had_df['Hadir'] == True]['Jam Sumbangan']
                   .apply(pd.to_numeric, errors='coerce').fillna(0).clip(lower=0).sum())
    kpi_hadir_rekod   = int((had_df['Hadir'] == True).sum())
    kpi_jumlah_rekod  = len(had_df)
    kpi_kadar_kehadiran = round(kpi_hadir_rekod / kpi_jumlah_rekod * 100, 1) \
        if kpi_jumlah_rekod else 0.0
else:
    kpi_jam = 0.0; kpi_kadar_kehadiran = 0.0

if not kew_df_sah.empty and 'Jenis' in kew_df_sah.columns:
    kpi_dana    = float(kew_df_sah[kew_df_sah['Jenis'].isin(
        ['Peruntukan','Sumbangan Luar'])]['Jumlah (RM)'].sum())
    kpi_belanja = float(kew_df_sah[kew_df_sah['Jenis'] == 'Perbelanjaan']['Jumlah (RM)'].sum())
else:
    kpi_dana = kpi_belanja = 0.0
kpi_baki = kpi_dana - kpi_belanja

kpi_laporan_hantar = int((prog_df['Laporan Dihantar'] == True).sum()) \
    if not prog_df.empty else 0
kpi_laporan_jumlah = len(prog_df) if not prog_df.empty else 0

print('✅ KPI berjaya dikira:')
print(f'   Program Selesai   : {kpi_program_selesai} / {SASARAN_PROGRAM} sasaran')
print(f'   Pelajar Terlibat  : {kpi_pelajar}')
print(f'   Pensyarah Terlibat: {kpi_pensyarah}')
print(f'   Penerima Manfaat  : {kpi_penerima:,}')
print(f'   Jam Sukarelawan   : {kpi_jam:,.1f} jam')
print(f'   Dana              : RM {kpi_dana:,.2f}')
print(f'   Perbelanjaan      : RM {kpi_belanja:,.2f}')
print(f'   Baki              : RM {kpi_baki:,.2f}')
print(f'   Laporan Dihantar  : {kpi_laporan_hantar} / {kpi_laporan_jumlah}')
print(f'   Kadar Kehadiran   : {kpi_kadar_kehadiran}%')

In [ ]:
# ── Sel 7: Jana carta ────────────────────────────────────────────────────────

def carta_kosong(tajuk):
    fig = go.Figure()
    fig.add_annotation(text='Tiada data untuk dipaparkan',
                       xref='paper', yref='paper', x=0.5, y=0.5,
                       showarrow=False, font=dict(size=14, color='#57606a'))
    fig.update_layout(title=tajuk, height=320,
                      plot_bgcolor='#f7f8fa', paper_bgcolor='#ffffff')
    return fig

def pendek(teks, n=30):
    return teks[:n]+'…' if isinstance(teks,str) and len(teks)>n else teks

# Carta 1 — Peserta hadir per program
if not had_df.empty and not prog_df.empty:
    agg1 = (had_df[had_df['Hadir']==True]
            .groupby('ID Program').size().reset_index(name='Peserta Hadir'))
    agg1 = agg1.merge(prog_df[['ID Program','Nama Program']], on='ID Program', how='left')
    agg1['Label'] = agg1['Nama Program'].fillna(agg1['ID Program']).apply(pendek)
    fig1 = px.bar(agg1, x='Label', y='Peserta Hadir',
                  title='Peserta Hadir mengikut Program',
                  labels={'Label':'Program','Peserta Hadir':'Bilangan Peserta'},
                  color_discrete_sequence=['#3b82d4'], text='Peserta Hadir')
    fig1.update_traces(textposition='outside')
    fig1.update_layout(xaxis_tickangle=-30, plot_bgcolor='#f7f8fa',
                       paper_bgcolor='#ffffff', height=350)
else:
    fig1 = carta_kosong('Peserta Hadir mengikut Program')

# Carta 2 — Kos per program
if not kew_df_sah.empty and not prog_df.empty:
    agg2 = (kew_df_sah[kew_df_sah['Jenis']=='Perbelanjaan']
            .groupby('ID Program')['Jumlah (RM)'].sum().reset_index(name='Perbelanjaan (RM)'))
    agg2 = agg2.merge(prog_df[['ID Program','Nama Program']], on='ID Program', how='left')
    agg2['Label'] = agg2['Nama Program'].fillna(agg2['ID Program']).apply(pendek)
    fig2 = px.bar(agg2, x='Label', y='Perbelanjaan (RM)',
                  title='Perbelanjaan (RM) mengikut Program',
                  labels={'Label':'Program','Perbelanjaan (RM)':'Jumlah (RM)'},
                  color_discrete_sequence=['#7c5cd8'], text='Perbelanjaan (RM)')
    fig2.update_traces(texttemplate='RM %{text:,.0f}', textposition='outside')
    fig2.update_layout(xaxis_tickangle=-30, plot_bgcolor='#f7f8fa',
                       paper_bgcolor='#ffffff', height=350)
else:
    fig2 = carta_kosong('Perbelanjaan (RM) mengikut Program')

# Carta 3 — Taburan status
if not prog_df.empty:
    agg3 = prog_df['Status'].value_counts().reset_index()
    agg3.columns = ['Status','Bilangan']
    fig3 = px.pie(agg3, names='Status', values='Bilangan',
                  title='Taburan Status Program',
                  color='Status', color_discrete_map=WARNA_STATUS, hole=0.45)
    fig3.update_traces(textinfo='label+percent')
    fig3.update_layout(paper_bgcolor='#ffffff', height=350,
                       legend=dict(orientation='h', yanchor='bottom', y=-0.2))
else:
    fig3 = carta_kosong('Taburan Status Program')

# Carta 4 — Jenis aktiviti
if not prog_df.empty and 'Jenis Aktiviti' in prog_df.columns:
    agg4 = prog_df['Jenis Aktiviti'].value_counts().reset_index()
    agg4.columns = ['Jenis Aktiviti','Bilangan']
    fig4 = px.bar(agg4, x='Bilangan', y='Jenis Aktiviti', orientation='h',
                  title='Program mengikut Jenis Aktiviti',
                  labels={'Bilangan':'Bilangan Program','Jenis Aktiviti':''},
                  color_discrete_sequence=['#22c55e'], text='Bilangan')
    fig4.update_traces(textposition='outside')
    fig4.update_layout(plot_bgcolor='#f7f8fa', paper_bgcolor='#ffffff',
                       height=350, yaxis=dict(autorange='reversed'))
else:
    fig4 = carta_kosong('Program mengikut Jenis Aktiviti')

print('✅ Empat carta berjaya dijana')

In [ ]:
# ── Sel 8: Bina jadual ringkasan ──────────────────────────────────────────────

if not prog_df.empty:
    jadual = prog_df[['ID Program','Nama Program','Tarikh','Jenis Aktiviti',
                       'Status','Penerima Manfaat','Laporan Dihantar',
                       'Pautan Laporan']].copy()

    # Agregat peserta hadir
    if not had_df.empty:
        pes = (had_df[had_df['Hadir']==True].groupby('ID Program')
               .size().reset_index(name='Peserta Hadir'))
        jam = (had_df[had_df['Hadir']==True].groupby('ID Program')['Jam Sumbangan']
               .sum().reset_index(name='Jam Sumbangan'))
        jadual = jadual.merge(pes, on='ID Program', how='left')
        jadual = jadual.merge(jam, on='ID Program', how='left')
    else:
        jadual['Peserta Hadir'] = 0
        jadual['Jam Sumbangan'] = 0

    # Agregat perbelanjaan
    if not kew_df_sah.empty:
        kos = (kew_df_sah[kew_df_sah['Jenis']=='Perbelanjaan']
               .groupby('ID Program')['Jumlah (RM)'].sum()
               .reset_index(name='Perbelanjaan (RM)'))
        jadual = jadual.merge(kos, on='ID Program', how='left')
    else:
        jadual['Perbelanjaan (RM)'] = 0

    # Isi NaN
    for lajur in ['Peserta Hadir','Jam Sumbangan','Perbelanjaan (RM)','Penerima Manfaat']:
        jadual[lajur] = jadual[lajur].fillna(0)

    # Format
    jadual['Laporan Dihantar'] = jadual['Laporan Dihantar'].map(
        {True:'✔ Ya', False:'✘ Tidak'}).fillna('✘ Tidak')
    jadual['Tarikh'] = pd.to_datetime(
        jadual['Tarikh'], errors='coerce').dt.strftime('%d/%m/%Y').fillna('—')

    print('✅ Jadual ringkasan berjaya dibina —', len(jadual), 'program')
else:
    jadual = pd.DataFrame()
    print('⚠️  Tiada data program untuk jadual ringkasan')

In [ ]:
# ── Sel 9: Papar dashboard dalam Colab ───────────────────────────────────────

def warna_status_html(status):
    return WARNA_STATUS.get(status, '#1f2328')

def bina_baris_jadual(baris):
    warna = warna_status_html(baris.get('Status',''))
    pautan = baris.get('Pautan Laporan','')
    pautan_html = (f'<a href="{pautan}" target="_blank">🔗</a>'
                   if pd.notna(pautan) and str(pautan).startswith('http') else '—')
    return f'''
    <tr>
      <td>{baris.get("ID Program","")}</td>
      <td style="max-width:180px">{baris.get("Nama Program","")}</td>
      <td>{baris.get("Tarikh","")}</td>
      <td>{baris.get("Jenis Aktiviti","")}</td>
      <td style="color:{warna};font-weight:600">{baris.get("Status","")}</td>
      <td style="text-align:center">{int(baris.get("Peserta Hadir",0))}</td>
      <td style="text-align:center">{baris.get("Jam Sumbangan",0):.0f}</td>
      <td style="text-align:center">{int(baris.get("Penerima Manfaat",0)):,}</td>
      <td style="text-align:right">RM {baris.get("Perbelanjaan (RM)",0):,.2f}</td>
      <td style="text-align:center">{baris.get("Laporan Dihantar","—")}</td>
      <td style="text-align:center">{pautan_html}</td>
    </tr>'''

baris_jadual = ''.join(
    bina_baris_jadual(r) for _, r in jadual.iterrows()
) if not jadual.empty else '<tr><td colspan="11" style="text-align:center">Tiada data</td></tr>'

# KPI delta
delta_prog = kpi_program_selesai - SASARAN_PROGRAM
delta_warna = '#22c55e' if delta_prog >= 0 else '#ef4444'
delta_tanda = f'+{delta_prog}' if delta_prog >= 0 else str(delta_prog)
baki_warna  = '#22c55e' if kpi_baki >= 0 else '#ef4444'

html_dashboard = f'''
<!DOCTYPE html>
<html lang="ms">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Dashboard CSR — JKE PKB</title>
<style>
  * {{ box-sizing: border-box; margin: 0; padding: 0; }}
  body {{ font-family: -apple-system,"Segoe UI",system-ui,sans-serif;
          font-size: 14px; background: #f7f8fa; color: #1f2328; line-height: 1.5; }}
  .header {{ background: #1f2328; color: #fff; padding: 18px 28px; }}
  .header h1 {{ font-size: 20px; }}
  .header p  {{ font-size: 13px; color: #adb5bd; margin-top: 2px; }}
  .badge {{ display:inline-block; background:#3b82d4; color:#fff;
            font-size:11px; padding:2px 8px; border-radius:10px; margin-left:8px; }}
  .container {{ max-width: 1200px; margin: 0 auto; padding: 20px; }}
  .section-title {{ font-size:16px; font-weight:700; margin: 24px 0 12px;
                    color:#1f2328; border-left:4px solid #3b82d4; padding-left:10px; }}
  .kpi-grid {{ display:grid; grid-template-columns:repeat(auto-fit,minmax(160px,1fr));
               gap:12px; margin-bottom:8px; }}
  .kpi-kad {{ background:#fff; border:1px solid #e5e7eb; border-radius:8px;
              padding:14px 12px; text-align:center; }}
  .kpi-label {{ font-size:11px; color:#57606a; margin-bottom:4px; }}
  .kpi-nilai {{ font-size:24px; font-weight:700; }}
  .kpi-sub   {{ font-size:11px; margin-top:2px; }}
  table {{ width:100%; border-collapse:collapse; background:#fff;
           border:1px solid #e5e7eb; border-radius:8px; overflow:hidden; font-size:13px; }}
  th {{ background:#f7f8fa; padding:10px 8px; text-align:left;
        border-bottom:2px solid #e5e7eb; font-size:12px; color:#57606a; }}
  td {{ padding:9px 8px; border-bottom:1px solid #f0f0f0; vertical-align:middle; }}
  tr:last-child td {{ border-bottom:none; }}
  tr:hover {{ background:#f7f8fa; }}
  .amaran {{ background:#fff8e1; border:1px solid #f59e0b; border-radius:6px;
             padding:10px 14px; margin-bottom:16px; font-size:13px; }}
  .footer {{ text-align:center; font-size:11px; color:#adb5bd;
             border-top:1px solid #e5e7eb; margin-top:40px; padding:16px; }}
</style>
</head>
<body>
<div class="header">
  <h1>⚡ Dashboard CSR — Jabatan Kejuruteraan Elektrik
    <span class="badge">{"Data Contoh" if "Contoh" in str(list(uploaded.keys())) else "Data Sebenar"}</span>
  </h1>
  <p>Politeknik Kota Bharu &nbsp;|&nbsp; {'🔒 Nama disembunyikan' if SEMBUNYIKAN_NAMA else '👁️ Nama dipapar'}</p>
</div>
<div class="container">

  {''.join(f"<div class=\"amaran\">⚠️ {a}</div>" for a in amaran) if amaran else ''}

  <div class="section-title">📊 Petunjuk Prestasi Utama (KPI)</div>
  <div class="kpi-grid">
    <div class="kpi-kad">
      <div class="kpi-label">✅ Program Selesai</div>
      <div class="kpi-nilai">{kpi_program_selesai}</div>
      <div class="kpi-sub" style="color:{delta_warna}">{delta_tanda} drpd sasaran {SASARAN_PROGRAM}</div>
    </div>
    <div class="kpi-kad">
      <div class="kpi-label">🎓 Pelajar Terlibat</div>
      <div class="kpi-nilai">{kpi_pelajar}</div>
    </div>
    <div class="kpi-kad">
      <div class="kpi-label">👨‍🏫 Pensyarah Terlibat</div>
      <div class="kpi-nilai">{kpi_pensyarah}</div>
    </div>
    <div class="kpi-kad">
      <div class="kpi-label">🤝 Penerima Manfaat</div>
      <div class="kpi-nilai">{kpi_penerima:,}</div>
    </div>
    <div class="kpi-kad">
      <div class="kpi-label">⏱️ Jam Sukarelawan</div>
      <div class="kpi-nilai">{kpi_jam:,.0f}</div>
      <div class="kpi-sub" style="color:#57606a">jam</div>
    </div>
    <div class="kpi-kad">
      <div class="kpi-label">💰 Jumlah Dana</div>
      <div class="kpi-nilai" style="font-size:18px">RM {kpi_dana:,.0f}</div>
    </div>
    <div class="kpi-kad">
      <div class="kpi-label">💸 Perbelanjaan</div>
      <div class="kpi-nilai" style="font-size:18px">RM {kpi_belanja:,.0f}</div>
    </div>
    <div class="kpi-kad">
      <div class="kpi-label">🏦 Baki Dana</div>
      <div class="kpi-nilai" style="font-size:18px;color:{baki_warna}">RM {kpi_baki:,.0f}</div>
      <div class="kpi-sub" style="color:{baki_warna}">{'✔ Surplus' if kpi_baki>=0 else '⚠ Defisit'}</div>
    </div>
    <div class="kpi-kad">
      <div class="kpi-label">📄 Laporan Dihantar</div>
      <div class="kpi-nilai">{kpi_laporan_hantar}/{kpi_laporan_jumlah}</div>
    </div>
    <div class="kpi-kad">
      <div class="kpi-label">📋 Kadar Kehadiran</div>
      <div class="kpi-nilai">{kpi_kadar_kehadiran}%</div>
    </div>
  </div>

  <div class="section-title">📋 Ringkasan Program CSR</div>
  <div style="overflow-x:auto">
  <table>
    <tr>
      <th>ID</th><th>Nama Program</th><th>Tarikh</th><th>Jenis Aktiviti</th>
      <th>Status</th><th>Peserta</th><th>Jam</th>
      <th>Penerima</th><th>Kos</th><th>Laporan</th><th>Pautan</th>
    </tr>
    {baris_jadual}
  </table>
  </div>

  <div class="section-title">📈 Carta & Visualisasi</div>
  <div id="carta1"></div>
  <div id="carta2"></div>
  <div id="carta3"></div>
  <div id="carta4"></div>

  <div class="footer">
    Dashboard CSR — Jabatan Kejuruteraan Elektrik, Politeknik Kota Bharu &nbsp;|&nbsp;
    Versi 1.0 &nbsp;|&nbsp; Untuk penggunaan dalaman sahaja
  </div>
</div>

{fig1.to_html(full_html=False, include_plotlyjs='cdn', div_id='carta1')}
{fig2.to_html(full_html=False, include_plotlyjs=False, div_id='carta2')}
{fig3.to_html(full_html=False, include_plotlyjs=False, div_id='carta3')}
{fig4.to_html(full_html=False, include_plotlyjs=False, div_id='carta4')}

</body></html>
'''

# Papar dalam Colab
display(HTML(html_dashboard))
print('✅ Dashboard dipapar di atas')

In [ ]:
# ── Sel 10: Muat turun dashboard.html ────────────────────────────────────────
# Jalankan sel ini untuk simpan dashboard sebagai fail HTML
# Fail boleh dibuka di mana-mana pelayar (Chrome/Edge) tanpa pemasangan

nama_fail_html = 'dashboard_csr_jke.html'

with open(nama_fail_html, 'w', encoding='utf-8') as f:
    f.write(html_dashboard)

print(f'✅ Fail "{nama_fail_html}" berjaya disimpan')
print('⬇️  Memulakan muat turun...')

colab_files.download(nama_fail_html)

print(f'\n📌 Cara guna fail HTML ini:')
print(f'   1. Salin fail ke PC jabatan (USB / emel)')
print(f'   2. Klik dua kali fail → terbuka dalam pelayar')
print(f'   3. Graf boleh dizum dan dihover seperti biasa')
print(f'   ⚠️  Nota: Graf memerlukan internet untuk muatkan Plotly.js')
print(f'      (Atau jalankan Sel 11 untuk versi tanpa internet)')

In [ ]:
# ── Sel 11 (OPSIONAL): Versi HTML tanpa internet ─────────────────────────────
# Jalankan HANYA jika anda mahu buka fail HTML di PC TANPA internet
# Fail akan lebih besar (~3MB) kerana Plotly.js disertakan terus

nama_fail_offline = 'dashboard_csr_jke_offline.html'

# Jana semula carta dengan Plotly.js embedded
html_offline = html_dashboard.replace(
    fig1.to_html(full_html=False, include_plotlyjs='cdn', div_id='carta1'),
    fig1.to_html(full_html=False, include_plotlyjs=True, div_id='carta1')
)
for fig, div_id in [(fig2,'carta2'),(fig3,'carta3'),(fig4,'carta4')]:
    html_offline = html_offline.replace(
        fig.to_html(full_html=False, include_plotlyjs=False, div_id=div_id),
        fig.to_html(full_html=False, include_plotlyjs=False, div_id=div_id)
    )

with open(nama_fail_offline, 'w', encoding='utf-8') as f:
    f.write(html_offline)

print(f'✅ Fail offline "{nama_fail_offline}" berjaya disimpan')
colab_files.download(nama_fail_offline)
print('⬇️  Muat turun dimulakan — fail ini berfungsi tanpa internet!')

In [ ]:
# ── Sel 12 (OPSIONAL): Muat turun CSV ringkasan ───────────────────────────────
# Jana dan muat turun jadual ringkasan sebagai CSV

if not jadual.empty:
    nama_csv = 'ringkasan_csr_jke.csv'
    jadual.to_csv(nama_csv, index=False, encoding='utf-8-sig')
    colab_files.download(nama_csv)
    print(f'✅ Jadual ringkasan berjaya dimuat turun sebagai "{nama_csv}"')
else:
    print('⚠️  Tiada jadual untuk dimuat turun')